# Factor Impact Exploration

## User Story

As a user, I want the system to show which environmental factor I should improve and how much my win probability may increase, so that I can adjust my environment before I play.

## Pipeline Overview

This trainer does not train a separate model. It uses the production win-rate model from `../trainer-winrate/models/model_pipeline.pkl` and performs what-if sensitivity analysis.

Run from `machine_learning/trainers/trainer-factor-imp`:

```bash
python steps/1_gather.py
python steps/2_features.py
python steps/3_analyze.py
python steps/4_evaluate.py
```

## Data Flow

1. `1_gather.py` reads `data/reworked_mock_data.csv` and creates `data/factor_examples.csv`.
2. `2_features.py` calculates `env_score` and creates one-factor candidates in `data/factor_candidates.csv`.
3. `3_analyze.py` loads the production win-rate model and writes `models/factor_impact_report.json`.
4. `4_evaluate.py` validates the report and writes `models/factor_impact_validation.json`.

## Model Inputs

The production win-rate model expects exactly these features:

- `minutes_slept`
- `minutes_awake`
- `env_score`
- `light`

The source dataset provides environmental columns under different names:

- `avg_lumen` -> `light`
- `avg_celsius` -> `temperature_celsius`
- `avg_ppm` -> `co2`
- `sleep_hours * 60` -> `minutes_slept`

`minutes_awake` is not available in the dataset, so the pipeline uses a documented default value of `60` minutes for now. `temperature_celsius` and `co2` are converted into `env_score` before prediction.

In [7]:
from pathlib import Path
import json
import pandas as pd

BASE_DIR = Path('..')
REPORT_PATH = BASE_DIR / 'models' / 'factor_impact_report.json'
VALIDATION_PATH = BASE_DIR / 'models' / 'factor_impact_validation.json'
EXAMPLES_PATH = BASE_DIR / 'data' / 'factor_examples.csv'
CANDIDATES_PATH = BASE_DIR / 'data' / 'factor_candidates.csv'

def calculate_env_score(temperature_celsius, co2):
    temp_dist = abs(temperature_celsius - 20)
    co2_norm = (co2 - 400) / 1600
    return 1 - (co2_norm * 0.7 + (temp_dist / 5) * 0.3)


## Generated Examples

`factor_examples.csv` contains the pre-game/environment inputs used by the analysis.

In [8]:
examples = pd.read_csv(EXAMPLES_PATH)
examples.head()


,example_name,minutes_slept,minutes_awake,temperature_celsius,co2,light,target
0,example_1,348.0,60,23.136253,564.998190,874.172214,0
1,example_2,450.0,60,18.673120,1844.084651,1911.285752,1
2,example_3,308.0,60,19.293030,1208.403796,1517.589095,0
3,example_4,590.0,60,25.188434,1722.331946,1277.585272,0
4,example_5,493.0,60,22.851432,912.079362,480.833553,1


## Factor-Impact Report

The report contains the current prediction, all candidate predictions, and the best positive recommendation for each example.

In [9]:
report = json.loads(REPORT_PATH.read_text())
validation = json.loads(VALIDATION_PATH.read_text())
validation


{'report_path': 'models/factor_impact_report.json',
 'valid': True,
 'n_examples': 200,
 'errors': []}

## Recommendation Summary

This table shows the selected recommendation for each example.

In [10]:
summary_rows = []

for example in report['examples']:
    best = example['best_recommendation']
    summary_rows.append({
        'example_name': example['example_name'],
        'current_win_probability': example['current_win_probability'],
        'recommended_factor': example['recommended_factor'],
        'current_value': None if best is None else best['current_value'],
        'recommended_value': None if best is None else best['recommended_value'],
        'improved_win_probability': None if best is None else best['win_probability'],
        'increase_percentage_points': None if best is None else best['increase_percentage_points'],
    })

summary = pd.DataFrame(summary_rows)
summary.head(20)


,example_name,current_win_probability,recommended_factor,current_value,recommended_value,improved_win_probability,increase_percentage_points
0,example_1,0.510,light,874.172214,1500.0,0.520,1.0
1,example_2,0.420,light,1911.285752,1500.0,0.490,7.0
2,example_3,0.450,temperature_celsius,19.293030,20.0,0.500,5.0
3,example_4,0.685,None,NaN,NaN,NaN,NaN
4,example_5,0.345,temperature_celsius,22.851432,20.0,0.430,8.5
5,example_6,0.610,None,NaN,NaN,NaN,NaN
6,example_7,0.580,None,NaN,NaN,NaN,NaN
7,example_8,0.435,light,1759.117062,1500.0,0.480,4.5
8,example_9,0.610,None,NaN,NaN,NaN,NaN
9,example_10,0.450,light,1474.530640,1500.0,0.455,0.5


## Candidate Details

Each example tests exactly one factor at a time:

- `temperature_celsius` -> `20`
- `co2` -> `500`
- `light` -> `1500`

In [11]:
candidate_rows = []

for example in report['examples']:
    for candidate in example['all_candidates']:
        candidate_rows.append({
            'example_name': example['example_name'],
            'factor': candidate['factor'],
            'current_value': candidate['current_value'],
            'recommended_value': candidate['recommended_value'],
            'current_win_probability': example['current_win_probability'],
            'candidate_win_probability': candidate['win_probability'],
            'increase_percentage_points': candidate['increase_percentage_points'],
        })

candidate_details = pd.DataFrame(candidate_rows)
candidate_details.head(30)


,example_name,factor,current_value,recommended_value,current_win_probability,candidate_win_probability,increase_percentage_points
0,example_1,temperature_celsius,23.136253,20.0,0.510,0.445,-6.5
1,example_1,co2,564.998190,500.0,0.510,0.465,-4.5
2,example_1,light,874.172214,1500.0,0.510,0.520,1.0
3,example_2,temperature_celsius,18.673120,20.0,0.420,0.350,-7.0
4,example_2,co2,1844.084651,500.0,0.420,0.230,-19.0
5,example_2,light,1911.285752,1500.0,0.420,0.490,7.0
6,example_3,temperature_celsius,19.293030,20.0,0.450,0.500,5.0
7,example_3,co2,1208.403796,500.0,0.450,0.460,1.0
8,example_3,light,1517.589095,1500.0,0.450,0.435,-1.5
9,example_4,temperature_celsius,25.188434,20.0,0.685,0.545,-14.0


## Limitations

- Recommendation quality depends on the quality of the production win-rate model.
- The current model may still be based on mock or limited data.
- `minutes_awake` is currently fixed to `60` because the source dataset does not provide it.
- These recommendations are model-based what-if results, not scientifically proven environmental advice.